# Pretrained Transformer Models (Hugging Face)
### Sprint 2, Item 2

Sprint 1 used `sentence-transformers` (built on Hugging Face models under the hood)
purely for embeddings. This notebook looks at the broader Hugging Face ecosystem:
the `transformers` library, model cards, and how embedding models differ from
classification-ready models — setting up the fine-tuning vs. zero-shot comparison
in the next notebook.


In [1]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. The `transformers` library vs `sentence-transformers`

- `sentence-transformers` wraps a transformer model with pooling logic specifically
  tuned to produce good sentence-level embeddings for similarity tasks.
- `transformers` (Hugging Face's core library) is lower-level: it gives you direct
  access to any model on the Hugging Face Hub — encoders, classifiers, generators —
  along with their tokenizers.

Both load models by name from the Hugging Face Hub; the difference is what you do
with the model's output.


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # same model Sprint 1 used

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

print(f"Loaded tokenizer and model for: {MODEL_NAME}")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

C:\Users\RanVic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 704.25it/s]


Loaded tokenizer and model for: sentence-transformers/all-MiniLM-L6-v2
Model has 22,713,216 parameters


## 2. Manually reproduce what `sentence-transformers` does internally
This shows the mean-pooling step that turns per-token outputs into one sentence
vector — useful to understand what's actually happening under `model.encode()`
in Sprint 1's `embeddings.ipynb`.

In [3]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch, seq_len, hidden_dim)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(1), min=1e-9)

sample_text = ["Experienced Python developer with machine learning background"]
encoded = tokenizer(sample_text, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    output = model(**encoded)

sentence_embedding = mean_pooling(output, encoded["attention_mask"])
sentence_embedding = torch.nn.functional.normalize(sentence_embedding, p=2, dim=1)

print("Manually-computed embedding shape:", sentence_embedding.shape)
print("First 5 values:", sentence_embedding[0][:5])

Manually-computed embedding shape: torch.Size([1, 384])
First 5 values: tensor([-0.0261, -0.0172,  0.0085,  0.0333,  0.0492])


## 3. Model cards: how to evaluate a model before using it
Before picking a model for a task, its Hugging Face model card typically documents:
training data, intended use, known limitations, and benchmark scores. Key things to
check for this project's resume-screening use case:
- Was it trained on general text, or domain-specific text (legal, medical, code)?
- What's its max input length (resumes can be long — check for truncation)?
- Are there documented biases (e.g. favoring certain phrasing/demographics)?


In [4]:
print(f"Model config summary for {MODEL_NAME}:")
print(f"  Max position embeddings (max input length): {model.config.max_position_embeddings}")
print(f"  Hidden size (embedding dimension): {model.config.hidden_size}")
print(f"  Number of attention heads: {model.config.num_attention_heads}")
print(f"  Number of hidden layers: {model.config.num_hidden_layers}")

Model config summary for sentence-transformers/all-MiniLM-L6-v2:
  Max position embeddings (max input length): 512
  Hidden size (embedding dimension): 384
  Number of attention heads: 12
  Number of hidden layers: 6


---
## Takeaway
`sentence-transformers` is a convenience layer over exactly this pattern: tokenize →
run through the transformer → pool the token outputs → normalize. Understanding this
matters for the next notebook, since zero-shot classification uses a *different* kind
of pretrained model — one with a classification head instead of just producing raw
embeddings.
